In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq

d:\LangGraph Tutorial\.langgraphvenv\lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
load_dotenv()

True

In [4]:
model = ChatGroq(model='openai/gpt-oss-120b')

In [5]:
# define a state

class PromptState(TypedDict):

    title: str
    outline: str
    blog: str
    evaluate: float

In [6]:
def create_outline(state: PromptState) -> PromptState:

    # grab a title
    title = state['title']

    # create a prompt
    prompt = f"create an outline for this topic -> {title}"

    # get the outline

    outline = model.invoke(prompt).content

    #update the state
    state['outline'] = outline

    return state

In [7]:
def create_blog(state: PromptState) -> PromptState:

    # grab a title and outline
    title = state['title']
    outline = state['outline']

    # create a prompt
    prompt = f"Generate a blog for the provided topic -> {title} and given outline -> {outline}"

    # get the blog
    blog = model.invoke(prompt).content

    # update the state
    state['blog'] = blog

    return state

In [8]:
def evaluate(state: PromptState) -> PromptState:

    # grab 
    title = state['blog']
    outline = state['outline']
    blog = state['blog']

    prompt = f"genarate a float score from 0 to 1 where close to 1 means accurate and close to 0 means not good to check if the given outline -> {outline} and blog -> {blog} matching or not"

    evaluate = model.invoke(prompt).content

    state['evaluate'] = evaluate

    return state

In [9]:
# define a graph
graph = StateGraph(PromptState)

# add a node
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate)

# add edges

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate')
graph.add_edge('evaluate',END)

# create a workflow

workflow = graph.compile()

In [10]:
input = {'title':'cristiano ronaldo'}

final_state = workflow.invoke(input)

In [11]:
print(final_state)

{'title': 'cristiano ronaldo', 'outline': '**Outline: Cristiano\u202fRonaldo – From Madeira’s Streets to Global Icon**\n\n---\n\n### I. Introduction  \n   A. Hook – a striking statistic or memorable moment (e.g., “300+ career goals for club and country combined”).  \n   B. Thesis statement – why Ronaldo is more than a footballer: a cultural phenomenon, a business brand, and a charitable force.  \n   C. Brief roadmap of the paper/presentation.\n\n---\n\n### II. Early Life & Foundations  \n   1. **Family background** – birth in Funchal, Madeira (5\u202fFeb\u202f1985); parents Maria\u202fDulce and José\u202fDinis.  \n   2. **Childhood influences** – love of football, early idolization of Luís\u202fFigo & Dutch “Total Football”.  \n   3. **Youth clubs** – Andorinha, Nacional, and the pivotal move to Sporting CP academy at 12.  \n   4. **Key traits developed** – work ethic, discipline, and the “Sii!” celebration origin.\n\n---\n\n### III. Club Career Chronology  \n   A. **Sporting CP (2002‑

In [14]:
for key, value in final_state.items():
    print(f"{key} -> {value}")
    print()

title -> cristiano ronaldo

outline -> **Outline: Cristiano Ronaldo – From Madeira’s Streets to Global Icon**

---

### I. Introduction  
   A. Hook – a striking statistic or memorable moment (e.g., “300+ career goals for club and country combined”).  
   B. Thesis statement – why Ronaldo is more than a footballer: a cultural phenomenon, a business brand, and a charitable force.  
   C. Brief roadmap of the paper/presentation.

---

### II. Early Life & Foundations  
   1. **Family background** – birth in Funchal, Madeira (5 Feb 1985); parents Maria Dulce and José Dinis.  
   2. **Childhood influences** – love of football, early idolization of Luís Figo & Dutch “Total Football”.  
   3. **Youth clubs** – Andorinha, Nacional, and the pivotal move to Sporting CP academy at 12.  
   4. **Key traits developed** – work ethic, discipline, and the “Sii!” celebration origin.

---

### III. Club Career Chronology  
   A. **Sporting CP (2002‑2003)**  
      - Debut, first professional goal, UEFA